# Advanced document indexing

# Splitting and ingesting HTML content

## Splitting and ingesting the content of a single URL (on Cornwall)

### Preparing the Chroma DB collections

In [1]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import getpass

/Users/basharnaji/Documents/GitHub/building-llm-applications/ch08/env_ch08/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
corwnall_granular_collection = Chroma( #A
    collection_name="cornwall_granular",
    embedding_function=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2'),
)

corwnall_granular_collection.reset_collection() #B
# A Create a Chorma DB collection
# B Reset the collection in case it already exists 

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [4]:
corwnall_coarse_collection = Chroma( #A 
    collection_name="cornwall_coarse",
    embedding_function=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2'),
)

corwnall_coarse_collection.reset_collection() #B
# A Create a Chorma DB collection
# B Reset the collection in case it already exists 

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Loading the HTML content with the AsyncHtmlLoader 

In [5]:
from langchain_community.document_loaders import AsyncHtmlLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [6]:
destination_url = "https://en.wikivoyage.org/wiki/Cornwall"

In [7]:
html_loader = AsyncHtmlLoader(destination_url)

In [8]:
docs = html_loader.load()

Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  2.73it/s]


In [9]:
len(docs)

1

### Splitting into granular chunks with the HTMLSectionSplitter 

In [10]:
from langchain_text_splitters import HTMLSectionSplitter

In [11]:
headers_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(
    headers_to_split_on=headers_to_split_on)

In [12]:
def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content #A
        temp_chunks = html_section_splitter.split_text(
            html_string) #B
        all_chunks.extend(temp_chunks) 

    return all_chunks

#A Extract the HTML text from the document
#B Each chunk is a H1 or H2 HTML section

In [13]:
granular_chunks = split_docs_into_granular_chunks(docs)

#### Ingesting granular chunks

In [14]:
corwnall_granular_collection.add_documents(documents=granular_chunks)

['bcd55420-b993-49e0-baee-af74ccddc8ec',
 '9fbdea20-3351-4d70-a08e-6ad694e95d4b',
 'c9c61bed-6413-4701-9ecf-823b8155e080',
 'd4d39e82-a087-442c-bc33-98474e01804a',
 '87998ade-0a41-458f-be4d-932425d60cec',
 'e8a4f2ea-01ff-4cf6-bc81-ac610dd11ff6',
 '921f4679-4130-40ba-83db-6193b2f018a9',
 '4c2a08ea-1bc1-4d36-affe-04d4291a9307',
 'b92736de-9717-4b44-940a-2aaf46c677fd',
 '97a77228-a3fe-4d3d-ad3c-1c531b1aa4dd',
 'dc9c888e-b095-4a40-8c9f-bd2812a0c9c9',
 '1cacb473-d23d-4c8b-8580-8e496cac35f6',
 'c8b7bbf2-53fa-496a-a3f3-328746354a00',
 '5985f6ee-9414-4a6b-a239-2858a6c10610',
 '04942847-efb1-4c20-a946-136f197eb67b',
 'ea7c4488-0643-4d5b-81df-a6e5b55b846b',
 '6bc30bdb-2ce3-4dee-bbfd-f4a3099bfcf7',
 'c39b43e7-f9ef-43c7-9304-63eb59be01de',
 '326da828-5981-457b-8002-b0f7cb1e7aee']

#### Searching granular chunks

In [15]:
results = corwnall_granular_collection.similarity_search(
    query="Events or festivals in Cornwall",k=3)
for doc in results:
    print(doc)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


page_content='Festivals 
 [ edit ] 
 
 These festivals tend to not be public holidays and not all are celebrated fully across the county. 
   
 AberFest .   A Celtic cultural festival celebrating “All things” Cornish and Breton that takes place biennially (every two years) in Cornwall at Easter. The AberFest Festival alternates with the Breizh – Kernow Festival that is held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate years.       ( updated Jun 2023 ) 
 Golowan , sometimes also  Goluan  or  Gol-Jowan  is the Cornish word for the Midsummer celebrations, most popular in the Penwith area and in particular  Penzance  and  Newlyn . The celebrations are conducted from the 23rd of June (St John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve being the more popular in Cornish fishing communities. The celebrations are centred around the lighting of bonfires and fireworks and the performance of associated rituals. Some towns have a street-parade dur

### Splitting into coarse chunks with the RecursiveCharacterTextSplitter 

In [16]:
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [17]:
html2text_transformer = Html2TextTransformer()

In [18]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000, chunk_overlap=300
)

In [19]:
def split_docs_into_coarse_chunks(docs):
    text_docs = html2text_transformer.transform_documents(
        docs) #A 
    coarse_chunks = text_splitter.split_documents(
        text_docs)

    return coarse_chunks
#A transform HTML docs into clean text docs

In [20]:
coarse_chunks = split_docs_into_coarse_chunks(docs)

#### Ingesting coarse chunks

In [21]:
corwnall_coarse_collection.add_documents(documents=coarse_chunks)

['fd2ec7bf-e561-419b-a413-00eeaf2daf95',
 '337d856a-f54b-4b74-8441-6ef32e98a572',
 'c29b9b1c-ba49-4715-94bb-9414611d0da7',
 'f1827216-b093-4e12-8bbc-97354ba57672',
 'b421461c-b823-4a66-8496-e5956228c535',
 'a70f34cb-097e-46d5-b23f-9945ccade98c',
 'e8fa493e-3eda-41a8-a3d8-55a2cc5a9da0',
 'd81826f2-cc3b-43b5-94bf-379cfc40eb27',
 'dcc36d47-3632-43ab-a467-1b11d19df307',
 '9797dde0-101c-40c6-9817-502526b5af8d',
 'ffaaf123-a28f-43b1-b179-1a4c44814fd6',
 '5037549f-a7f9-4441-a8d0-0aa094f18303',
 '002dd0bb-4f5e-4f16-9c33-f3fb232d333f',
 'd21761bc-a858-4d01-9c76-b1020736912e',
 '81230690-89bb-419a-b0f0-f6bc6726d633']

#### Searching coarse chunks

In [22]:
results = corwnall_coarse_collection.similarity_search(
    query="Events or festivals in Cornwall",k=3)
for doc in results:
    print(doc)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


page_content='### Spirits

[edit]

    _See also:Liquor_

Gin and rum are also produced in Cornwall. A popular brand of Cornish rum is
Dead Man's Fingers which has multiple flavours and is bottled in St. Ives.

## Festivals

[edit]

These festivals tend to not be public holidays and not all are celebrated
fully across the county.

AberFest. A Celtic cultural festival celebrating “All things” Cornish and
Breton that takes place biennially (every two years) in Cornwall at Easter.
The AberFest Festival alternates with the Breizh – Kernow Festival that is
held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate
years.  (updated Jun 2023)

**Golowan** , sometimes also _Goluan_ or _Gol-Jowan_ is the Cornish word for
the Midsummer celebrations, most popular in the Penwith area and in particular
Penzance and Newlyn. The celebrations are conducted from the 23rd of June (St
John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve
being the more popular in Corn

## Splitting and ingesting the content of various URLs (across UK destinations)

### Preparing the Chroma DB collections

In [23]:
uk_granular_collection = Chroma( #A
    collection_name="uk_granular",
    embedding_function=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
)

uk_granular_collection.reset_collection() #B

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [24]:
uk_coarse_collection = Chroma( #A
    collection_name="uk_coarse",
    embedding_function=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
)

uk_coarse_collection.reset_collection() #B

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Splitting and ingesting HTML content with the HTMLSectionSplitter 

In [25]:
# Reduce this list if you want to save on processing fees
uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall", 
    "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
    "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven"
    "East_Sussex", "Brighton", "Battle", "Hastings_(England)", 
    "Rye_(England)", "Seaford", "Ashdown_Forest"
] 

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"

In [26]:
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' for d in uk_destinations]

In [27]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #C
    docs =  html_loader.load() #D
    
    for doc in docs:
        print(doc.metadata)
        granular_chunks = split_docs_into_granular_chunks(docs)
        uk_granular_collection.add_documents(documents=granular_chunks)

        coarse_chunks = split_docs_into_coarse_chunks(docs)
        uk_coarse_collection.add_documents(documents=coarse_chunks)
#A Create a Chroma DB collection
#B Reset the collection in case it already exists 
#C Loader for one destination
#D Documents of one destination 

Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  2.68it/s]


{'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  3.20it/s]


{'source': 'https://en.wikivoyage.org/wiki/North_Cornwall', 'title': 'North Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  3.15it/s]


...

{'source': 'https://en.wikivoyage.org/wiki/Rye_(England)', 'title': 'Rye (England) – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  3.08it/s]


{'source': 'https://en.wikivoyage.org/wiki/Seaford', 'title': 'Seaford – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  2.90it/s]


{'source': 'https://en.wikivoyage.org/wiki/Ashdown_Forest', 'title': 'Ashdown Forest – Travel guide at Wikivoyage', 'language': 'en'}


#### Searching 

In [28]:
granular_results = uk_granular_collection.similarity_search(
    query="Events or festivals in East Sussex",k=4)
for doc in granular_results:
    print(doc)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


page_content='Festivals 
 [ edit ] 
 
 These festivals tend to not be public holidays and not all are celebrated fully across the county. 
   
 AberFest .   A Celtic cultural festival celebrating “All things” Cornish and Breton that takes place biennially (every two years) in Cornwall at Easter. The AberFest Festival alternates with the Breizh – Kernow Festival that is held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate years.       ( updated Jun 2023 ) 
 Golowan , sometimes also  Goluan  or  Gol-Jowan  is the Cornish word for the Midsummer celebrations, most popular in the Penwith area and in particular  Penzance  and  Newlyn . The celebrations are conducted from the 23rd of June (St John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve being the more popular in Cornish fishing communities. The celebrations are centred around the lighting of bonfires and fireworks and the performance of associated rituals. Some towns have a street-parade dur

In [29]:
coarse_results = uk_coarse_collection.similarity_search(
    query="Events or festivals in East Sussex",k=4)
for doc in coarse_results:
    print(doc)

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


page_content='### Events

[edit]

A market during the Brighton Festival. The Theatre Royal is the red building.
A colourful parade down Queens Road during Pride in 2016.

  * **Brighton Racecourse** has flat-racing April-Oct. It's on Freshfield Rd a mile east of town centre.
  * **Plumpton Racecourse** is National Hunt (jumps races) Nov-March, but it's 10 mi (16 km) north in Lewes.
  * Brighton Festival Fringe: early May – early June, ☏ +44 1273 764900, info@brightonfringe.org. The Fringe runs at the same time as the main festival, and features over 600 events, including comedy, theatre, music, and "open houses" (local artists exhibiting in their own homes) and tours (haunted pubs, Regency Brighton, churches, cemeteries, sewers, etc.)_ (date needs fixing)_
  * Brighton Festival: May, ☏ +44 1273 709709 (tickets), tickets@brightonfestival.org. The Brighton Festival, in May each year, is the second biggest arts festival in Great Britain (coming closely behind Edinburgh). Music of all sort

In [30]:
granular_results = uk_granular_collection.similarity_search(
    query="Beaches in Conrwall",k=4)
for doc in granular_results:
    print(doc)

page_content='Do 
 [ edit ] 
 
 
 
 
   
 Cornish Film Festival .   Held annually for two weeks each November around  Newquay .       ( updated Jan 2024 ) 
 
 
 
 
 50.415741 -5.091478 1   Newquay Golf Club ,   Tower Road, TR7 1LT ,   ☏   +44 1637 872091 ,   info@newquaygolfclub.co.uk .   9AM-4PM .   A semi-private golf club established in 1890. Total yardage Championship: 6141, Men: 5708, and Women: 5364. £31 for non-members.     ( updated Apr 2019 ) 
 
 
 
 Beaches 
 [ edit ] 
 
 Fistral Beach 
 Newquay is well known as a surfer's paradise. Therefore it offers plenty of beaches: 
 
 
 Crantock Beach  - quiet beach, 2   km away from the city centre along the coastal path 
 
 Fistral Beach  - Newquays most popular beach, located to the west of Towards Head. Famous as a surf centre, has life guards during summer months. International surf competitions are held here. 
 
 Great Western - a popular family beach, can be accessed from Cliff Road besides to the Great Western Hotel. 
 
 Harbou

In [30]:
coarse_results = uk_coarse_collection.similarity_search(
    query="Beaches in Cornwall",k=4)
for doc in coarse_results:
    print(doc)

page_content='**South Cornwall** is in Cornwall. It includes much of the stunning Cornish
coast along the English Channel of the Atlantic Ocean.

## Towns and villages

[edit]

Map of South Cornwall

  * 50.26-5.0511 Truro — Cornwall's main centre hosts the Royal Cornwall Museum
  * 50.3311-4.20212 Cawsand — overlooks Plymouth Sound; Cawsand is within Mount Edgcumbe Country Park
  * 50.15-5.073 Falmouth — famous for its beaches, it is home to the world's third largest natural harbour
  * 50.334-4.6334 Fowey — the Fowey Regatta in mid-August attracts many yachts and sailing boats
  * 50.354-4.4545 Looe — a summer resort place with a monkey sanctuary, and an active fishing village
  * 50.408-4.2126 Saltash — "Gateway to Cornwall", a small town on the Cornwall side of the Tamar crossings
  * 50.338-4.7957 St Austell — largest town in the county and home to the Eden Project, the world's largest greenhouse
  * 50.3314-4.75788 Charlestown — seaside town used as filming location for the TV sh

# Embedding strategy

## Embedding child chunks with ParentDocumentRetriever

In [32]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Setting up the Parent Document retriever

In [33]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500) #B

child_chunks_collection = Chroma( #C
    collection_name="uk_child_chunks",
    embedding_function=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)

child_chunks_collection.reset_collection() #D

doc_store = InMemoryStore() #E

parent_doc_retriever = ParentDocumentRetriever( #F
    vectorstore=child_chunks_collection,
    docstore=doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Splitter to generate child granular chunks from parent coarse chunks
#C Vector store collection to host child granular chunks
#D Make sure the collection is empty
#E Document store to host parent coarse chunks
#F Retriever to link parent coarse chunks to child granular chunks

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Ingesting the content into doc and vector store

In [34]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(html_docs) #C

    print(f'Ingesting {destination_url}')
    parent_doc_retriever.add_documents(text_docs, ids=None) #D

#A Loader for destination web page
#B HTML documents of one destination 
#C Transform HTML docs into clean text deocs
#D Ingest coarse chunks into document store and granular chunks into vector store

Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  2.05it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  3.62it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  4.63it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  4.53it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  4.18it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [35]:
list(doc_store.yield_keys())
#A Show the keys of the added coarse chunks

['cac0783b-699f-4bb4-a605-737d4775d8e1',
 'a73f126b-53ac-4a91-87a9-b848588a7269',
 'ab57bda3-e305-4b84-920b-4fe56d9a3538',
 'db441e72-6ed1-40f9-a7dd-af286d2ce394',
 '4ff0c3d0-59e3-480a-931f-f9833170077d',
 '0733b6d5-a91e-4f3f-9986-6d28aa2967ad',
 '5ba1caa7-56c6-4067-bb6e-4f6414913a2c',
 '49606cb6-8b2e-475c-acfb-f6ccdf40ce8e',
 'ad5831a9-e8a4-41bc-8ad4-7e38b77d78de',
 'bab25331-5453-4856-91a6-cbea6fc307c2',
 '827a568a-06e3-4af8-bf29-4275a574df8a',
 'bdf86e21-e326-4ccf-a508-85194d58b258',
 '0ec9f3e6-c2c4-4747-a9a6-d25370a7abe6',
 '1a46d39b-fac6-4a3d-9a01-dfc82c9cb0f4',
 '17460117-1985-4355-80ad-a16237984cc0',
 'f91b556e-7d36-449d-9671-6a1fa34b1d83',
 'a81ca605-f987-4c09-90ec-733557443464',
 'b40847a8-495b-42fc-b8e7-c14b2e3a47b5',
 '2bbe6f6a-fc28-410c-85d2-858c105999c3',
 'bbb4a0df-fd2b-48ee-a6ee-5d511382d837',
 '52fd2910-f48d-49fc-8749-65369575a9e4',
 'bf78fa53-cd7d-446d-9fab-0cd4ccd0328c',
 '8cfd2421-2662-49ed-995c-2456019cffba',
 'cce37707-19ee-4b36-bc11-506f39aa1d01',
 'e1a7a07a-59f3-

### Performing a search on granular information 

In [36]:
retrieved_docs = parent_doc_retriever.invoke("Cornwall Ranger")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [37]:
len(retrieved_docs)

4

In [38]:
retrieved_docs[0]

Document(metadata={'source': 'https://en.wikivoyage.org/wiki/St_Ives', 'title': 'St Ives – Travel guide at Wikivoyage', 'language': 'en'}, page_content='[edit]\n\nThe **Tourist Information Centre** is on Street-an-Pol, TR26 2DS +44 1736\n796297. It\'s open all year round and can help with accommodation.\n\n## Get in\n\n[edit]\n\n### By plane\n\n[edit]\n\n**Cornwall Newquay Airport** (**NQY**  IATA) has flights from London (Gatwick\n& Stansted), Dublin, Leeds, Bradford, Bristol, Manchester and Cardiff. It\'s\nabout 30 miles (48 km) away from St Ives by A3075 and A30, reckon 45 min by\ncar. Its two disadvantages are i) most flights are summer only; ii) public\ntransport is tricky as first you have to get the bus into Newquay (last bus\naround 6:30PM), then rely on an infrequent bus or branch-line train for\nconnections to St Ives.\n\n**Exeter Airport** (**EXT**  IATA) is further but may be a simpler option. It\nhas more flights year-round, the airport bus runs until 10PM, and mainline\nt

### Comparing with direct semantic search on child chunks

In [39]:
child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")

In [40]:
len(child_docs_only)

4

In [41]:
child_docs_only[0]

Document(id='55ba5269-041c-42a9-8061-d33ca3f41218', metadata={'doc_id': '7b94d431-88a7-42ec-825f-111a52a2c266', 'language': 'en', 'source': 'https://en.wikivoyage.org/wiki/St_Ives', 'title': 'St Ives – Travel guide at Wikivoyage'}, page_content='Other nearby destinations along the main line, served by hourly trains from\nPlymouth, are Penzance (for Newlyn, Mousehole and Isles of Scilly), St Austell\n(for Eden Project), Par (for Newquay), and Bodmin. See National Rail for times\nand fares; advance booking is usually much cheaper. A _Ride Cornwall Ranger_\nis good value for local travel. It allows unlimited off-peak travel within\nCornwall, and between Cornwall and Plymouth, by all trains and most buses.')

In [42]:
# IMPORTANT: as you can see a granular search would have identified the chunk, but it would have lost the usefulcontext about travelling in Cornwall

## Embedding child chunks with MultiVectorRetriever

In [43]:
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

### Setting up the Multi vector retriever

In [44]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000) #A
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500) #B

child_chunks_collection = Chroma( #C
    collection_name="uk_child_chunks",
    embedding_function=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
)

child_chunks_collection.reset_collection() #D

doc_byte_store = InMemoryByteStore() #E
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #F
    vectorstore=child_chunks_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Splitter to generate child granular chunks from parent coarse chunks
#C Vector store collection to host child granular chunks
#D Make sure the collection is empty
#E Document store to host parent coarse chunks
#F Retriever to link parent coarse chunks to child granular chunks

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Ingesting the content into doc and vector store

In [45]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_granular_chunks = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E
        
        coarse_chunk_id = coarse_chunks_ids[i]
            
        granular_chunks = child_splitter.split_documents(
            [coarse_chunk]) #F

        for granular_chunk in granular_chunks:
            granular_chunk.metadata[doc_key] = coarse_chunk_id #G

        all_granular_chunks.extend(granular_chunks)

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_granular_chunks) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination 
#C transform HTML docs into clean text docs
#D Split the destination content into parent coarse chunks
#E Iterate over the parent coarse chunks
#F Create child granular chunks form each parent coarse chunk
#G Link each child granular chunk to its parent coarse chunk
#H Ingest the child granular chunks into the vector store
#I Ingest the parent coarse chunks into the document store

Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  2.39it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  2.88it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  4.55it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  2.96it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  3.11it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


### Performing a search on granular information

In [46]:
retrieved_docs = multi_vector_retriever.invoke(
    "Cornwall Ranger")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [47]:
len(retrieved_docs)

4

In [48]:
retrieved_docs[0]

Document(metadata={'source': 'https://en.wikivoyage.org/wiki/St_Ives', 'title': 'St Ives – Travel guide at Wikivoyage', 'language': 'en'}, page_content='[edit]\n\nThe **Tourist Information Centre** is on Street-an-Pol, TR26 2DS +44 1736\n796297. It\'s open all year round and can help with accommodation.\n\n## Get in\n\n[edit]\n\n### By plane\n\n[edit]\n\n**Cornwall Newquay Airport** (**NQY**  IATA) has flights from London (Gatwick\n& Stansted), Dublin, Leeds, Bradford, Bristol, Manchester and Cardiff. It\'s\nabout 30 miles (48 km) away from St Ives by A3075 and A30, reckon 45 min by\ncar. Its two disadvantages are i) most flights are summer only; ii) public\ntransport is tricky as first you have to get the bus into Newquay (last bus\naround 6:30PM), then rely on an infrequent bus or branch-line train for\nconnections to St Ives.\n\n**Exeter Airport** (**EXT**  IATA) is further but may be a simpler option. It\nhas more flights year-round, the airport bus runs until 10PM, and mainline\nt

In [49]:
##IMPORTANT: same as Parent Document retriever, but more control and flexibility on how to link child to parent chunks

### Comparing with direct semantic search on child chunks

In [50]:
child_docs_only =  child_chunks_collection.similarity_search(
    "Cornwall Ranger")

In [51]:
len(child_docs_only)

4

In [52]:
child_docs_only[0]

Document(id='913efd64-2d40-4649-941d-a5262abf3902', metadata={'doc_id': '57f3d8a5-7f67-4ca2-9470-f9059160cb62', 'language': 'en', 'source': 'https://en.wikivoyage.org/wiki/St_Ives', 'title': 'St Ives – Travel guide at Wikivoyage'}, page_content='Other nearby destinations along the main line, served by hourly trains from\nPlymouth, are Penzance (for Newlyn, Mousehole and Isles of Scilly), St Austell\n(for Eden Project), Par (for Newquay), and Bodmin. See National Rail for times\nand fares; advance booking is usually much cheaper. A _Ride Cornwall Ranger_\nis good value for local travel. It allows unlimited off-peak travel within\nCornwall, and between Cornwall and Plymouth, by all trains and most buses.')

In [65]:
## IMPORTANT: Same as before

## Embedding summaries with MultiVectorRetriever

In [56]:
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_anthropic import ChatAnthropic
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid

### Setting up the Multi vector retriever (similar to when embedding child chunks)

In [72]:
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000) #A

summaries_collection = Chroma( #B
    collection_name="uk_summaries",
    embedding_function=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
)

summaries_collection.reset_collection() #C

doc_byte_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=summaries_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host parent coarse chunks
#E Retriever to link parent coarse chunks to child granular chunks

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Setting up the summarization chain

In [60]:
ANTHROPIC_API_KEY = getpass.getpass('Enter your ANTHROPIC_API_KEY')
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", api_key=ANTHROPIC_API_KEY)

Enter your ANTHROPIC_API_KEY ········


In [61]:
summarization_chain = (
    {"document": lambda x: x.page_content} #A
    | ChatPromptTemplate.from_template("Summarize the following document:\n\n{document}") #B
    | llm
    | StrOutputParser())

#A Grab the text content from the document
#B Instantiate a prompt asking to generate summary of the provided text
#C Send the LLM the instantiated prompt 
#D Extract the summary text from the response

### Ingesting the coarse chunks and related summaries into doc and vector store

In [62]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_summaries = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E
        
        coarse_chunk_id = coarse_chunks_ids[i]
            
        summary_text =  summarization_chain.invoke(
            coarse_chunk) #F
        summary_doc = Document(page_content=summary_text, 
                               metadata={doc_key: coarse_chunk_id})

        all_summaries.append(summary_doc) #G

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_summaries) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination 
#C transform HTML docs into clean text docs
#D Split the destination content into coarse chunks
#E Iterate over the coarse chunks
#F Generate a summary for the coarse chunk thorugh the summarization chain
#G Link each summary to its related coarse chunk
#H Ingest the summaries into the vector store
#I Ingest the coarse chunks into the document store

Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  3.17it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  4.26it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  4.54it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  4.32it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  4.07it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [63]:
# COMMENT: the code above is similar to when ingesting child chunks, but it is slower because of the summarization step
# which invokes the LLM.
# The processing can be speeded up by parallelizing the outer for loop on the destination urls.

### Performing a search on granular information

In [64]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall travel")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [65]:
len(retrieved_docs)

4

In [66]:
retrieved_docs

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/South_Cornwall', 'title': 'South Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content="## Get around\n\n[edit]\n\n### By bus\n\n[edit]\n\nThanks to Transport for Cornwall, all bus tickets are interchangeable across\nthe different companies. The **Cornwall All Day ticket** allows unlimited\ntravel for a calendar day. As of 2023, fares are £5 for adults and £4 for\nunder-19s. Payment is by cash or contactless. The two main bus companies are:\n\n  * **Go Cornwall Bus** covers all parts of Cornwall and connects with Plymouth (in Devon).\n  * **Kernow** (part of First Bus) covers western and central Cornwall.\n\nBuses only serve designated stops when in towns; otherwise, you can flag them\ndown anywhere that's safe for them to stop.\n\n### By train\n\n[edit]\n\n**CrossCountry Trains** and **Great Western Railway** operate regular train\nservices between the main centres of population, the latter company also\ns

### Comparing with direct semantic search on summaries

In [67]:
summary_docs_only =  summaries_collection.similarity_search(
    "Cornwall Travel")

In [68]:
len(summary_docs_only)

4

In [69]:
summary_docs_only

[Document(id='5672b449-e15d-40f0-846d-8b8dcf2d5668', metadata={'doc_id': 'a8d2231e-40bc-4cef-a2ce-8f57076039c7'}, page_content="# Summary\n\nThis document provides a travel and tourism guide for Cornwall:\n\n## Getting Around\n- **Bus**: Transport for Cornwall offers interchangeable tickets across companies. The Cornwall All Day ticket costs £5 (adults) or £4 (under-19s) for unlimited daily travel. Two main operators are Go Cornwall Bus and Kernow.\n- **Train**: CrossCountry Trains and Great Western Railway serve major towns. The Cornwall Ranger ticket (£14 adults, £7 under-16s) allows unlimited daily travel in Cornwall and Plymouth.\n\n## Main Attractions\n- **Eden Project** - Two large domes near St Austell showcasing global flora and a zip line\n- **Lost Gardens of Heligan** - 80-acre landscaped gardens near Mevagissey\n- **National Maritime Museum** in Falmouth\n- **National Trust properties** including Antony House, Cotehele, Trelissick, and Glendurgan—historic estates with garden

In [70]:
# COMMENT: a direct search on summaries retrieves denser information, but it is missing out on useful details. 
# However, you might consider using the summaries directly if after testing they prove adequate.

## Embedding hypothetical questions with MultiVectorRetriever

In [71]:
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_anthropic import ChatAnthropic
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
import uuid
from typing import List
from pydantic import BaseModel, Field

### Setting up the Multi vector retriever (same as when embedding summaries)

In [74]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000) #A

hypothetical_questions_collection = Chroma( #B
    collection_name="uk_hypothetical_questions",
    embedding_function=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
)

hypothetical_questions_collection.reset_collection() #C

doc_byte_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=hypothetical_questions_collection,
    byte_store=doc_byte_store
)
#A Splitter to generate parent coarse chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host parent coarse chunks
#E Retriever to link parent coarse chunks to child granular chunks

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Setting up the chain to generate hypothetical questions

In [75]:
class HypotheticalQuestions(BaseModel):
    """A list of hypotetical questions for given text."""

    questions: List[str] = Field(..., description="List of hypothetical questions for given text")

In [76]:
llm_with_structured_output = ChatAnthropic(
    model="claude-haiku-4-5-20251001", 
    api_key=ANTHROPIC_API_KEY).with_structured_output(
        HypotheticalQuestions
)

In [77]:
hypothetical_questions_chain = (
    {"document_text": lambda x: x.page_content} #A
    | ChatPromptTemplate.from_template( #B
        "Generate a list of exactly 4 hypothetical questions that the below text could be used to answer:\n\n{document_text}"
    )
    | llm_with_structured_output #C
    | (lambda x: x.questions) #D
)

#A Grab the text content from the document
#B Instantiate a prompt asking to generate 4 hypothetical questions on the provided text
#C Invoke the LLM configured to return an object containing the questions as a typed list of strings
#D Grab the list of questions from the response

### Ingesting the coarse chunks and related hypothetical questions into doc and vector store

In [78]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    coarse_chunks = parent_splitter.split_documents(
        text_docs) #D

    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    all_hypothetical_questions = []
    for i, coarse_chunk in enumerate(
        coarse_chunks): #E
        
        coarse_chunk_id = coarse_chunks_ids[i]
            
        hypothetical_questions = hypothetical_questions_chain.invoke(
            coarse_chunk) #F
        hypothetical_questions_docs = [Document(
            page_content=question, metadata={doc_key: coarse_chunk_id})
                    for question 
                    in hypothetical_questions] #G

        all_hypothetical_questions.extend(hypothetical_questions_docs)

    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        all_hypothetical_questions) #H
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks))) #I

#A Loader for one destination
#B Documents of one destination 
#C transform HTML docs into clean text docs
#D Split the destination content into coarse chunks
#E Iterate over the coarse chunks
#F Generate a list of hypothetical questions for the coarse chunk thorugh the question generation chain
#G Link each hypothetical question to its related coarse chunk
#H Ingest the hypothetical questions into the vector store
#I Ingest the coarse chunks into the document store

Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  3.19it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  3.23it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  4.48it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  3.22it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  4.17it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


### Performing a search on granular information

In [79]:
retrieved_docs = multi_vector_retriever.invoke(
    "How can you go to Brighton from London?")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [80]:
len(retrieved_docs)

4

In [81]:
retrieved_docs

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/Brighton', 'title': 'Brighton – Travel guide at Wikivoyage', 'language': 'en'}, page_content='Brighton  \n---  \nClimate chart (explanation)  \n| J| F| M| A| M| J| J| A| S| O| N| D  \n---|---|---|---|---|---|---|---|---|---|---|---  \n      74     8 4 |        57     9 3 |        42     11 4 |        34     14 5 |        45     17 9 |        39     19 11 |        50     21 13 |        55     21 13 |        47     19 11 |        77     16 9 |        86     12 6 |        75     9 4  \nAverage max. and min. temperatures in °C  \nPrecipitation+Snow totals in mm  \nSource: Wikipedia. Visit the Met Office for a five day forecast.  \n| Imperial conversion  \n---  \nJ| F| M| A| M| J| J| A| S| O| N| D  \n      2.9     46 39 |        2.2     47 38 |        1.7     51 39 |        1.3     56 42 |        1.8     62 47 |        1.5     67 52 |        2     70 56 |        2.2     70 56 |        1.9     67 52 |        3     61 48 |        3

### Inspecting possible questions matching our question through semantic search

In [82]:
hypothetical_question_docs_only = hypothetical_questions_collection.similarity_search(
    "How can you go to Brighton from London?")

In [83]:
len(hypothetical_question_docs_only)

4

In [84]:
hypothetical_question_docs_only

[Document(id='776a65e2-39bc-4c60-a1f1-67b1136b61ef', metadata={'doc_id': '4b3467a3-0640-4f6d-bebb-9fc05504d6a7'}, page_content='How can someone travel to Brighton by train from London and other nearby cities?'),
 Document(id='688c6dc3-4fc1-481d-9cef-f86fb7357d65', metadata={'doc_id': 'c299f6ea-4298-4162-bf06-b4629130b3a7'}, page_content='What are the main transportation options for getting to and around Brighton?'),
 Document(id='7afd1608-b958-4270-80c8-1a864b84f3bc', metadata={'doc_id': 'd1e130b5-b580-4bd9-bb97-c530bd0fc361'}, page_content='How can visitors reach Brighton from Gatwick Airport and what is the travel time?'),
 Document(id='c41171ef-4dd6-44f6-81b8-63a473b02a1a', metadata={'doc_id': 'e6a5bc50-049c-441c-abbc-d5bf8bf33e85'}, page_content='What destinations can be accessed from Brighton traveling east, west, north, and south?')]

# Granular chunk expansion with MultiVectorRetriever

In [85]:
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

### Setting up the Multi vector retriever

In [87]:
granular_chunk_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500) #A

granular_chunks_collection = Chroma( #B
    collection_name="uk_granular_chunks",
    embedding_function=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
)

granular_chunks_collection.reset_collection() #C

expanded_chunk_store = InMemoryByteStore() #D
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever( #E
    vectorstore=granular_chunks_collection,
    byte_store=expanded_chunk_store
)
#A Splitter to generate granular chunks from original documents (parsed from web pages)
#B Vector store collection to host child granular chunks
#C Make sure the collection is empty
#D Document store to host expanded chunks
#E Retriever to link parent coarse chunks to child granular chunks

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


### Ingesting granular and expanded chunks into doc and vector store

In [88]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url) #A
    html_docs =  html_loader.load() #B
    text_docs = html2text_transformer.transform_documents(
        html_docs) #C

    granular_chunks = granular_chunk_splitter.split_documents(
        text_docs) #D

    expanded_chunk_store_items = []
    for i, granular_chunk in enumerate(
        granular_chunks): #E

        this_chunk_num = i #F
        previous_chunk_num = i-1 #F
        next_chunk_num = i+1 #F
        
        if i==0: #F
            previous_chunk_num = None
        elif i==(len(granular_chunks)-1): #F
            next_chunk_num = None

        expanded_chunk_text = "" #G
        if previous_chunk_num: #G
            expanded_chunk_text += granular_chunks[
                previous_chunk_num].page_content
            expanded_chunk_text += "\n"

        expanded_chunk_text += granular_chunks[
            this_chunk_num].page_content #G
        expanded_chunk_text += "\n"

        if next_chunk_num: #G
            expanded_chunk_text += granular_chunks[
                next_chunk_num].page_content
            expanded_chunk_text += "\n"

        expanded_chunk_id = str(uuid.uuid4()) #H
        expanded_chunk_doc = Document(
            page_content=expanded_chunk_text) #I

        expanded_chunk_store_item = (expanded_chunk_id, 
                                     expanded_chunk_doc)
        expanded_chunk_store_items.append(
            expanded_chunk_store_item)

        granular_chunk.metadata[
            doc_key] = expanded_chunk_id #J
            
    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(
        granular_chunks) #K
    multi_vector_retriever.docstore.mset(
        expanded_chunk_store_items) #L

#A Loader for one destination
#B Documents of one destination 
#C transform HTML docs into clean text docs
#D Split the destination content into granular chunks
#E Iterate over the granular chunks
#F determine the index of the current chunk and its previous and next chunks
#G Assemble the text of the expanded chunk by including the previous and next chunk
#H Generate the ID of the expanded chunk
#I Create the expanded chunk document
#J Link each granular chunk to its related expanded chunk
#K Ingest the granular chunks into the vector store
#L Ingest the expanded chunks into the document store

Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  3.10it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  4.42it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  4.57it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  4.55it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|###############################################################################################################| 1/1 [00:00<00:00,  4.16it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


### Performing a search on granular information

In [89]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall Ranger")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


In [90]:
len(retrieved_docs)

4

In [91]:
retrieved_docs[0]

Document(metadata={}, page_content='An overnight sleeper train, _The Night Riviera_ , runs between London\nPaddington and Penzance, stopping at St Erth. It runs Sunday to Friday from\nLondon Paddington around 11:30PM, reaching St Erth before 8AM (on Sunday\nbefore 9AM Monday). The return train leaves Penzance Su-F around 9:30PM,\npicking up at St Erth 10 min later, to reach London Paddington at 5AM; you can\nstay aboard until 7AM. Book via Great Western, with airline seats, or single\nor double sleeper cabins available.\nOther nearby destinations along the main line, served by hourly trains from\nPlymouth, are Penzance (for Newlyn, Mousehole and Isles of Scilly), St Austell\n(for Eden Project), Par (for Newquay), and Bodmin. See National Rail for times\nand fares; advance booking is usually much cheaper. A _Ride Cornwall Ranger_\nis good value for local travel. It allows unlimited off-peak travel within\nCornwall, and between Cornwall and Plymouth, by all trains and most buses.\nis goo

### Comparing with direct semantic search on granular chunks

In [92]:
child_docs_only =  child_chunks_collection.similarity_search("Cornwall Ranger")

In [93]:
len(child_docs_only)

4

In [94]:
child_docs_only[0]

Document(id='913efd64-2d40-4649-941d-a5262abf3902', metadata={'doc_id': '57f3d8a5-7f67-4ca2-9470-f9059160cb62', 'language': 'en', 'source': 'https://en.wikivoyage.org/wiki/St_Ives', 'title': 'St Ives – Travel guide at Wikivoyage'}, page_content='Other nearby destinations along the main line, served by hourly trains from\nPlymouth, are Penzance (for Newlyn, Mousehole and Isles of Scilly), St Austell\n(for Eden Project), Par (for Newquay), and Bodmin. See National Rail for times\nand fares; advance booking is usually much cheaper. A _Ride Cornwall Ranger_\nis good value for local travel. It allows unlimited off-peak travel within\nCornwall, and between Cornwall and Plymouth, by all trains and most buses.')

In [95]:
# COMMENT: the expanded chunk has more useful context